In [92]:
# Imports
import pandas as pd
import time as t
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)

In [ ]:
# Fix paths for repo

# Weather- and Biddingzonedata Germany
de_bz_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/BiddingZones/DE-LU_BZ.csv")
de_we_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Weatherdata/DE_Wetter.csv")
de_ep_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Generationdata/PG_DE.csv")
de_bz_data["country"] = "DE"
de_we_data["country"] = "DE"
de_ep_data["country"] = "DE"

# Weather- and Biddingzonedata Poland
pl_bz_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/BiddingZones/PL_BZ.csv")
pl_we_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Weatherdata/PL_Wetter.csv")
pl_ep_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Generationdata/PG_PL.csv")
pl_bz_data["country"] = "PL"
pl_we_data["country"] = "PL"
pl_ep_data["country"] = "PL"

# Weather- and Biddingzonedata Finnland
fi_bz_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/BiddingZones/FI_BZ.csv")
fi_we_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Weatherdata/FI_Wetter.csv")
fi_ep_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Generationdata/PG_FI.csv")
fi_bz_data["country"] = "FI"
fi_we_data["country"] = "FI"
fi_ep_data["country"] = "FI"

# Weather- and Biddingzonedata Czech Republic
cz_bz_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/BiddingZones/CZ_BZ.csv")
cz_we_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Weatherdata/CZ_Wetter.csv")
cz_ep_data = pd.read_csv("C:/Users/nicoh/Desktop/DSProject/Generationdata/PG_CZ.csv")
cz_bz_data["country"] = "CZ"
cz_we_data["country"] = "CZ"
cz_ep_data["country"] = "CZ"

In [94]:
# Combination of raw data into big dataframe s
we_data = pd.concat([de_we_data, pl_we_data, fi_we_data, cz_we_data])
bz_data = pd.concat([de_bz_data, pl_bz_data, fi_bz_data, cz_bz_data])
ep_data = pd.concat([de_ep_data, pl_ep_data, fi_ep_data, cz_ep_data])

In [95]:
# Convert timestamp columns into true timestamp columns so years, months and days can be extracted
bz_data["timestamp"] = pd.to_datetime(bz_data["timestamp"], utc=True)
we_data["timestamp"] = pd.to_datetime(we_data["time"], utc=True)
ep_data["timestamp"] = pd.to_datetime(ep_data["timestamp"], utc=True)

bz_data["year"] = bz_data["timestamp"].dt.year
we_data["year"] = we_data["timestamp"].dt.year
ep_data["year"] = ep_data["timestamp"].dt.year

bz_data["month"] = bz_data["timestamp"].dt.month
we_data["month"] = we_data["timestamp"].dt.month
ep_data["month"] = ep_data["timestamp"].dt.month

bz_data["day"] = bz_data["timestamp"].dt.day
we_data["day"] = we_data["timestamp"].dt.day
ep_data["day"] = ep_data["timestamp"].dt.day

In [96]:
# Simplify the nameing of the relevant column in the price data
bz_data.columns = bz_data.columns.str.replace("values.", "", regex=False)

In [97]:
# Filter for common timewindows and to full hours
bz_data_filtered = bz_data[(bz_data["year"] >= 2019) & (bz_data["year"] < 2025)]
we_data_filtered = we_data[(we_data["year"] >= 2019) & (we_data["year"] < 2025)]
ep_data_filtered = ep_data[(ep_data["year"] >= 2019) & (ep_data["year"] < 2025) & (ep_data["timestamp"].dt.minute == 0)] # Production data is sometimes in 15 miute intervals, since its in MW (Megawats) we can simplyfy by just looking at the full hour which then is equal to the MWh (Megawats/hour)

In [98]:
# Fill NaN with 0, only necessary for Energyproduction since some countries have production types none of the others use
ep_data_filtered = ep_data_filtered.fillna(0)

In [99]:
# Calculate Sums for different generation types
ep_data_filtered["generation_sum"] = ep_data_filtered["Nuclear"] \
    + ep_data_filtered["Hydro Run-of-River"] \
    + ep_data_filtered["Biomass"] \
    + ep_data_filtered["Fossil brown coal / lignite"] \
    + ep_data_filtered["Fossil hard coal"] \
    + ep_data_filtered["Fossil oil"] \
    + ep_data_filtered["Fossil coal-derived gas"] \
    + ep_data_filtered["Fossil gas"] \
    + ep_data_filtered["Fossil peat"] \
    + ep_data_filtered["Geothermal"] \
    + ep_data_filtered["Hydro water reservoir"] \
    + ep_data_filtered["Hydro pumped storage"] \
    + ep_data_filtered["Others"] \
    + ep_data_filtered["Waste"] \
    + ep_data_filtered["Wind offshore"] \
    + ep_data_filtered["Wind onshore"] \
    + ep_data_filtered["Solar"] 

ep_data_filtered["generation_fossil_sum"] = ep_data_filtered["Fossil brown coal / lignite"] \
    + ep_data_filtered["Fossil hard coal"] \
    + ep_data_filtered["Fossil oil"] \
    + ep_data_filtered["Fossil coal-derived gas"] \
    + ep_data_filtered["Fossil gas"] \
    + ep_data_filtered["Fossil peat"] \
    + ep_data_filtered["Waste"] 

ep_data_filtered["generation_renew_sum"] = ep_data_filtered["Hydro Run-of-River"] \
    + ep_data_filtered["Biomass"] \
    + ep_data_filtered["Geothermal"] \
    + ep_data_filtered["Hydro water reservoir"] \
    + ep_data_filtered["Hydro pumped storage"] \
    + ep_data_filtered["Wind offshore"] \
    + ep_data_filtered["Wind onshore"] \
    + ep_data_filtered["Solar"] \
    + ep_data_filtered["Others"] \
    + ep_data_filtered["Other renewables"]

ep_data_filtered["wind"] = ep_data_filtered["Wind offshore"] + ep_data_filtered["Wind onshore"]

ep_data_filtered["water"] = ep_data_filtered["Hydro Run-of-River"] + ep_data_filtered["Hydro water reservoir"] + ep_data_filtered["Hydro pumped storage"]

In [100]:
# Weather-, Energyproduction- and Biddingzonedate aggregation for each country and day
we_agg = (
    we_data_filtered
    .groupby(["year", "month", "day", "country"])
    .agg(
        temperature_daily_mean          = ("temperature_2m_max", "mean")
        ,shortwave_radiation_daily_mean = ("shortwave_radiation_sum", "mean")
        ,windspeed_daily_mean           = ("wind_speed_10m_mean", "mean")
        ,soil_moisture_daily_mean       = ("soil_moisture_28_to_100cm_mean", "mean")
        )
    .reset_index()
)

bz_agg = (
    bz_data_filtered
    .groupby(["year", "month", "day", "country"])["day_ahead_price"]
    .agg(
        price_mean  = "mean",
        price_std   = "std",
        price_min   = "min",
        price_max   = "max",
    )
    .reset_index()
)

ep_agg = (
    ep_data_filtered
    .groupby(["year", "month", "day", "country"])
    .agg(
        total_generation_mean   = ("generation_sum", "mean")
        ,total_generation_std   = ("generation_sum", "std")
        ,total_generation_sum   = ("generation_sum", "sum")

        ,solar_generation_mean  = ("Solar", "mean")
        ,solar_generation_std   = ("Solar", "std")
        ,solar_generation_sum   = ("Solar", "sum")     

        ,wind_generation_mean   = ("wind", "mean")
        ,wind_generation_std    = ("wind", "std")
        ,wind_generation_sum    = ("wind", "sum")

        ,water_generation_mean  = ("water", "mean")
        ,water_generation_std   = ("water", "std")
        ,water_generation_sum   = ("water", "sum")

        ,fossil_generation_mean = ("generation_fossil_sum", "mean")
        ,fossil_generation_std  = ("generation_fossil_sum", "std")
        ,fossil_generation_sum  = ("generation_fossil_sum", "sum")

        ,renew_generation_mean  = ("generation_renew_sum", "mean")
        ,renew_generation_std   = ("generation_renew_sum", "std")
        ,renew_generation_sum   = ("generation_renew_sum", "sum")
    )
    .reset_index()
)

In [101]:
# Combination of everything into large dataframe
daily_pre = pd.merge(
    we_agg
    ,bz_agg
    ,on     = ["year", "month", "day", "country"]
    ,how    = "inner"
)

daily = pd.merge(
    daily_pre
    ,ep_agg
    ,on     = ["year", "month", "day", "country"]
    ,how    = "inner"
)

In [102]:
# Calculating thresholds for "extreme weather", this part was created with LLM assistence
country_list = []
for c in ["DE", "FI", "PL", "CZ"]:
    temp = daily[daily["country"] == c]

    for i in range(1,13):
        m = temp[daily["month"] == i]
        val = [c, i, float(m["temperature_daily_mean"].quantile(0.90)), float(m["windspeed_daily_mean"].quantile(0.10)), float(m["shortwave_radiation_daily_mean"].quantile(0.10)), float(m["soil_moisture_daily_mean"].quantile(0.10))]
        country_list.append(val)

threshold = pd.DataFrame(country_list, columns=["country", "month", "temp_90", "wind_10", "solar_10", "soil_10"])

C:\Users\nicoh\AppData\Local\Temp\ipykernel_16848\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_16848\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_16848\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_16848\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_16848\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_16848\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match

In [103]:
# Combination into large dataframe
daily_enriched = pd.merge(
    daily
    ,threshold
    ,on     = ["month", "country"]
    ,how    = "inner"
)

In [104]:
# Determining weather event candiataes by comparing again the threshold
daily_enriched["high_heat_candidate"] = (daily_enriched["temperature_daily_mean"]           >= daily_enriched["temp_90"])
daily_enriched["low_wind_candidate"]  = (daily_enriched["windspeed_daily_mean"]             <= daily_enriched["wind_10"])
daily_enriched["low_solar_candidate"] = (daily_enriched["shortwave_radiation_daily_mean"]   <= daily_enriched["solar_10"])
daily_enriched["low_soil_candidate"]  = (daily_enriched["soil_moisture_daily_mean"]         <= daily_enriched["soil_10"])

In [105]:
# Find unusual weather periods (3 day periods), LLM used to figure out how to compare groups of entries for consecutive candidates
daily_enriched = daily_enriched.sort_values(["country", "year", "month", "day"])

# High heat (3 days)
daily_enriched["high_heat"] = (
    daily_enriched["high_heat_candidate"]
    &   (
        (daily_enriched["high_heat_candidate"].shift(1, fill_value=False) & daily_enriched["high_heat_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["high_heat_candidate"].shift(1, fill_value=False) & daily_enriched["high_heat_candidate"].shift(2, fill_value=False))
        | (daily_enriched["high_heat_candidate"].shift(-1, fill_value=False) & daily_enriched["high_heat_candidate"].shift(-2, fill_value=False))
        )
    )

# Low heat (3 days)
daily_enriched["low_wind"] = (
    daily_enriched["low_wind_candidate"]
    &   (
        (daily_enriched["low_wind_candidate"].shift(1, fill_value=False) & daily_enriched["low_wind_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["low_wind_candidate"].shift(1, fill_value=False) & daily_enriched["low_wind_candidate"].shift(2, fill_value=False))
        | (daily_enriched["low_wind_candidate"].shift(-1, fill_value=False) & daily_enriched["low_wind_candidate"].shift(-2, fill_value=False))
        )
    )

# Low solar (3 days)
daily_enriched["low_solar"] = (
    daily_enriched["low_solar_candidate"]
    &   (
        (daily_enriched["low_solar_candidate"].shift(1, fill_value=False) & daily_enriched["low_solar_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["low_solar_candidate"].shift(1, fill_value=False) & daily_enriched["low_solar_candidate"].shift(2, fill_value=False))
        | (daily_enriched["low_solar_candidate"].shift(-1, fill_value=False) & daily_enriched["low_solar_candidate"].shift(-2, fill_value=False))
        )
    )

# Low soil (3 days)
daily_enriched["low_soil"] = (
    daily_enriched["low_soil_candidate"]
    &   (
        (daily_enriched["low_soil_candidate"].shift(1, fill_value=False) & daily_enriched["low_soil_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["low_soil_candidate"].shift(1, fill_value=False) & daily_enriched["low_soil_candidate"].shift(2, fill_value=False))
        | (daily_enriched["low_soil_candidate"].shift(-1, fill_value=False) & daily_enriched["low_soil_candidate"].shift(-2, fill_value=False))
        )
    )

In [106]:
# Calculate the Renewable Share
daily_enriched["renew_share"] = daily_enriched["renew_generation_sum"] / daily_enriched["total_generation_sum"] * 100

In [107]:
# Calulating prices and renewable shares, split for weather events and normal days so it can later be compared 

# Calculate means for low wind events
daily_enriched_agg_wind = (
    daily_enriched
    .groupby(["country", "year", "low_wind", "month"])
    .agg(
        price_std_mean      = ("price_std","mean")
        ,renew_share_mean   = ("renew_share","mean")
    )
    .reset_index()
)

# Combine extrem and normal data for low wind events per year and month
daily_enriched_agg_wind_compact = pd.merge(
    daily_enriched_agg_wind[daily_enriched_agg_wind["low_wind"]]
    ,daily_enriched_agg_wind[~daily_enriched_agg_wind["low_wind"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["low_wind_extreme","low_wind_normal"], axis=1)
daily_enriched_agg_wind_compact["event_type"] = "low_wind"

# Calculate means for heatwave events
daily_enriched_agg_heat = (
    daily_enriched
    .groupby(["country", "year", "high_heat", "month"])
    .agg(
        price_std_mean      = ("price_std","mean")
        ,renew_share_mean   = ("renew_share","mean")
    )
    .reset_index()
)

# Combine extrem and normal data for heatwave events per year and month
daily_enriched_agg_heat_compact = pd.merge(
    daily_enriched_agg_heat[daily_enriched_agg_heat["high_heat"]]
    ,daily_enriched_agg_heat[~daily_enriched_agg_heat["high_heat"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["high_heat_extreme","high_heat_normal"], axis=1)
daily_enriched_agg_heat_compact["event_type"] = "high_heat"

# Calculate means for low solar events
daily_enriched_agg_solar = (
    daily_enriched
    .groupby(["country", "year", "low_solar", "month"])
    .agg(
        price_std_mean      = ("price_std","mean")
        ,renew_share_mean   = ("renew_share","mean")
    )
    .reset_index()
)

# Combine extrem and normal data for low solar events per year and month
daily_enriched_agg_solar_compact = pd.merge(
    daily_enriched_agg_solar[daily_enriched_agg_solar["low_solar"]]
    ,daily_enriched_agg_solar[~daily_enriched_agg_solar["low_solar"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["low_solar_extreme","low_solar_normal"], axis=1)
daily_enriched_agg_solar_compact["event_type"] = "low_solar"

# Calculate means for soilmoisture events
daily_enriched_agg_soil = (
    daily_enriched
    .groupby(["country", "year", "low_soil", "month"])
    .agg(
        price_std_mean      = ("price_std","mean")
        ,renew_share_mean   = ("renew_share","mean")
    )
    .reset_index()
)

# Combine extrem and normal data for low soilmoisture events per year and month
daily_enriched_agg_soil_compact = pd.merge(
    daily_enriched_agg_soil[daily_enriched_agg_soil["low_soil"]]
    ,daily_enriched_agg_soil[~daily_enriched_agg_soil["low_soil"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["low_soil_extreme","low_soil_normal"], axis=1)
daily_enriched_agg_soil_compact["event_type"] = "low_soil"

In [ ]:
# Combine into one large dataframe
daily_enriched_agg = (pd.concat([daily_enriched_agg_wind_compact,daily_enriched_agg_heat_compact,daily_enriched_agg_solar_compact,daily_enriched_agg_soil_compact]))

In [110]:
# Last calculation and renamings and removing unneeded columns
daily_enriched_agg["price_std_mean_diff_percentage"] = (daily_enriched_agg["price_std_mean_extreme"] - daily_enriched_agg["price_std_mean_normal"]) / daily_enriched_agg["price_std_mean_normal"] * 100
daily_enriched_agg["renew_share"] = daily_enriched_agg["renew_share_mean_extreme"]
daily_enriched_agg = daily_enriched_agg.drop(["price_std_mean_extreme","price_std_mean_normal","renew_share_mean_extreme","renew_share_mean_normal"], axis=1)

In [114]:
# Data for Visual 1 and 2
daily_enriched_agg.to_csv("Price_Volatility_Renewshare.csv")

# Data for Visual 3
daily_pre.to_csv("Price_Volatility_Overall.csv")